# 作业 02：根据三个线索，自己完成训练

请参考作业 01，独立编写完整的 PyTorch 训练流程，让下面的预测函数尽量拟合给定数据。

本题只给出三个线索。参数如何创建、如何计算梯度以及如何更新，都由你完成。可以查阅作业 01，但请根据本题的预测函数调整代码。

## 线索 1：预测函数

这次有三个待学习参数 $w_1,w_2,w_3$：

$$\hat y=w_1x+w_2x^2+w_3.$$

请根据这个公式，自己编写计算预测值的代码。输入整个 `x` 张量时，应得到每个样本对应的预测值。三个待学习参数也由你创建。请保留这个预测公式。

其中，$w_1$ 是一次项的系数，$w_2$ 是二次项的系数，$w_3$ 是常数项。三个参数在训练中都需要被调整。

## 线索 2：损失函数

每个样本的损失仍然是预测误差的绝对值：

$$L_i=|y_i-\hat y_i|.$$

本次有多个样本，请将所有样本的损失取平均，得到用于训练的一个数：

$$L=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

这里 $N$ 是样本数。请自己用 PyTorch 写出这个损失，不要换成平方误差。

## 线索 3：两个数据张量

`x[i]` 和 `y[i]` 是同一个样本的输入和目标。训练时保持这些数据不变，不要直接填写你猜测的参数作为答案。

import torch
# 输入数据与对应的目标数据，均为浮点数
w1=torch.tensor(0.0, requires_grad=True)
w2=torch.tensor(0.0, requires_grad=True)
w3=torch.tensor(0.0, requires_grad=True)

y_pred = w1*x + w2*x**2 + w3
loss = torch.abs(y - y_pred)

x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

## 你的作答区

请在下面编写完整训练代码，可以自行增加单元格。训练结束后，展示学到的三个参数、每个样本的预测值，以及最终的平均绝对误差。最终展示的预测值和损失应使用训练结束后的参数计算。

再用文字说明：你对作业 01 的训练代码做了哪些修改？你如何判断这次训练有效？不要求参数或误差恰好等于某个数，也不要求每一轮损失都下降。

In [1]:
# 在这里编写你的训练代码import torch

import torch

x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

# 输入数据与对应的目标数据，均为浮点数
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
w3 = torch.tensor(0.0, requires_grad=True)

y_pred = w1 * x + w2 * (x ** 2) + w3
loss = torch.mean(torch.abs(y - y_pred))

optimizer = torch.optim.SGD([w1, w2, w3], lr=0.1)
optimizer.zero_grad()
loss = torch.mean(torch.abs(y - y_pred))
loss.backward()

print("更新前的 w1, w2, w3: ", w1.item(), w2.item(), w3.item())
print("更新前的损失: ", loss.item())
print("梯度: ", w1.grad.item(), w2.grad.item(), w3.grad.item())

optimizer.step()

with torch.no_grad():
    y_pred_new = w1 * x + w2 * (x ** 2) + w3
    new_loss = torch.mean(torch.abs(y - y_pred_new))
    print("更新后的 w: ", w1.item(), w2.item(), w3.item())
    print("更新后的损失: ", new_loss.item())


def train_w(initial_w=1.0, epochs=1000, base_lr=0.1, decay=True):
    w1 = torch.tensor(float(initial_w), requires_grad=True)
    w2 = torch.tensor(float(initial_w), requires_grad=True)
    w3 = torch.tensor(float(initial_w), requires_grad=True)
    optimizer = torch.optim.SGD([w1, w2, w3], lr=base_lr)
    history = []
    for epoch in range(1, epochs + 1):
        lr = base_lr / epoch if decay else base_lr
        optimizer.param_groups[0]["lr"] = lr
        optimizer.zero_grad()
        y_pred_new = w1 * x + w2 * (x ** 2) + w3
        loss = torch.mean(torch.abs(y - y_pred_new))
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            y_pred_new = w1 * x + w2 * (x ** 2) + w3
            current_loss = torch.mean(torch.abs(y - y_pred_new)).item()
            history.append((epoch, lr, w1.item(), w2.item(), w3.item(), current_loss))
    return w1.detach(), w2.detach(), w3.detach(), history


learned_w1, learned_w2, learned_w3, history = train_w()
print("轮数    学习率        w1         w2         w3         损失")
for index in [0, 1, 4, 9, 99, 999]:
    epoch, lr, w1v, w2v, w3v, current_loss = history[index]
    print(f"{epoch:4d}  {lr:10.6f}  {w1v:10.6f}  {w2v:10.6f}  {w3v:10.6f}  {current_loss:10.6f}")

with torch.no_grad():
    final_pred = learned_w1 * x + learned_w2 * (x ** 2) + learned_w3
    final_mae = torch.mean(torch.abs(y - final_pred))

print("\n训练结束：")
print("w1 =", learned_w1.item())
print("w2 =", learned_w2.item())
print("w3 =", learned_w3.item())
print("每个样本预测值：", final_pred)
print("最终平均绝对误差MAE：", final_mae.item())

更新前的 w1, w2, w3:  0.0 0.0 0.0
更新前的损失:  2.1388888359069824
梯度:  -0.8888888955116272 -0.1666666567325592 -0.4444444477558136
更新后的 w:  0.08888889104127884 0.01666666567325592 0.04444444552063942
更新后的损失:  2.0404319763183594
轮数    学习率        w1         w2         w3         损失
   1    0.100000    1.011111    0.855556    0.977778    1.802469
   2    0.050000    1.022222    0.788889    0.972222    1.710494
   5    0.020000    1.039630    0.684444    0.963519    1.566399
  10    0.010000    1.053977    0.598360    0.956345    1.447634
 100    0.001000    1.104164    0.297239    0.931251    1.032198
1000    0.000100    1.183404    0.114636    0.978868    0.821520

训练结束：
w1 = 1.1834039688110352
w2 = 0.11463580280542374
w3 = 0.9788675308227539
每个样本预测值： tensor([-0.9294, -0.5383, -0.0899,  0.4158,  0.9789,  1.5992,  2.2769,  3.0119,
         3.8042])
最终平均绝对误差MAE： 0.8215202689170837


### 我的观察与说明

请在这里填写你的回答。

本次模型公式为 $\hat y = w_1 x + w_2 x^2 + w_3$，相比上一次作业的单参数二次模型，新增了一次项系数 $w_1$ 和常数项 $w_3$，一共三个参数一起用SGD优化，损失函数依旧使用平均绝对误差MAE。

观察训练过程：随着轮数增加，学习率不断衰减，损失整体持续下降，从一开始的2.13逐步降到0.82左右，参数w1、w2、w3慢慢收敛稳定，说明训练有效。MAE损失函数在零点处不可导，所以损失曲线会存在小幅震荡，不需要每一轮都严格下降。

训练结束得到参数：
w1 ≈ 1.1834，w2 ≈ 0.1146，w3 ≈ 0.9787
用这组参数对全部样本做预测，最终平均绝对误差MAE≈0.8215。

## 提交

重启内核并从上到下运行全部单元格，确认代码可以完整运行。保存包含代码、输出和文字回答的本文件，再提交到自己的 GitHub 仓库。